# 🎮 MLBB PlayStore Review — Sentiment Analysis Preprocessing Pipeline

**Tujuan:** Membersihkan data noisy dari review MLBB di PlayStore untuk keperluan analisis sentimen.

**Stack Library:**
| Tujuan | Library |
|--------|---------|
| Text Cleaning | `re`, `emoji`, custom function |
| Slang Normalization | Custom dictionary + Sastrawi |
| Stemming | `PySastrawi` |
| Stopwords | `Sastrawi` + `nltk` |
| Sentiment Lexicon | `VADER` (baseline) + IndoBERT |
| Modeling | IndoBERT (`indobenchmark/indobert-base-p1`) |

**Kolom Dataset:**
- `review` → teks ulasan (main target)
- `rating` → integer 1–5 (ground truth)
- `tanggal` → YYYY-MM-DD HH:MM:SS

**Jenis Noise yang Ditangani:**
- Typo & Spelling Error
- Slang & Singkatan
- Kata Berulang
- Emoji & Special Character
- Kata Kasar
- Code-Mixing (Bahasa Indonesia–Inggris)

---
## 📦 CELL 1 — Instalasi Library

In [ ]:
# ============================================================
# CELL 1 — Instalasi Library
# Jalankan sekali. Restart kernel jika diminta.
# ============================================================
!pip install PySastrawi emoji vaderSentiment transformers torch nltk pandas numpy matplotlib seaborn scikit-learn tqdm -q

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

print('✅ Semua library berhasil diinstall.')

---
## 📚 CELL 2 — Import Library

In [ ]:
# ============================================================
# CELL 2 — Import Library
# ============================================================
import pandas as pd
import numpy as np
import re
import emoji
import json
import os
import warnings
from tqdm import tqdm

# NLP
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Transformers (IndoBERT)
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

# Visualisasi
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import Counter

warnings.filterwarnings('ignore')
tqdm.pandas()

# Konfigurasi tampilan
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_columns', 20)
matplotlib.rcParams['figure.dpi'] = 120

print('✅ Semua library berhasil diimport.')
print(f'   PyTorch  : {torch.__version__}')
print(f'   GPU/CPU  : {"GPU (CUDA)" if torch.cuda.is_available() else "CPU"}')

---
## 📂 CELL 3 — Load Dataset

In [ ]:
# ============================================================
# CELL 3 — Load Dataset
# Ganti path sesuai lokasi file CSV Anda
# ============================================================

CSV_PATH = 'data_mlbb_reviews.csv'  # <-- GANTI dengan path file Anda

# --------------------
# Jika file belum ada, buat sample data demo berdasarkan contoh nyata
# --------------------
if not os.path.exists(CSV_PATH):
    print('⚠️  File CSV tidak ditemukan. Membuat sample data demo...')
    sample_reviews = [
        {'review': 'milih tim nya aneh ngak sesuai sama lawan malah tambah ngak jelass!!', 'rating': 1, 'tanggal': '2026-05-24 13:12:40'},
        {'review': 'plis lah monton gw cape anikin reng tapi knp di kasih tim bot semua. masa saya setiap hari main ml harus party biar menang', 'rating': 2, 'tanggal': '2026-05-23 10:05:22'},
        {'review': 'saya menyukainya', 'rating': 5, 'tanggal': '2026-05-22 08:30:11'},
        {'review': 'maer kang', 'rating': 3, 'tanggal': '2026-05-21 15:44:30'},
        {'review': 'ini hp aku yg bermasalah apa gimana? masa aku mau update yg ver allstar gak bisa aku dah coba ke play store tetep GK bisa tulisannya mainkan sama uninstall gak ada kata updateteya .😭😭Padahal kan aku mau mancing ikan fyp ku tentang omg mancing di ml terus kan aku jadi fomo😭😭', 'rating': 2, 'tanggal': '2026-05-20 19:22:05'},
        {'review': 'hei kepada pihak MONTOON tolong lah jika saat gameplay ada sinyal lag itu jangan nurunin bintang lah jangan bilang soal paket data/wifi jelek saya sudah menggunakan sinyal wifi 15mbps masih pada saat di coba main game sebelah itu bagus saat di pake main ml itu lag terus gimana ini tolong di jawab', 'rating': 2, 'tanggal': '2026-05-20 11:10:00'},
        {'review': 'Dear Moonton, tolong perbaiki sistem matchmaking yang lebih seimbang. Player yang serius dan bermain baik sering dipasangkan dengan teammate yang tidak setara, sehingga pengalaman bermain jadi kurang menyenangkan. Semoga matchmaking bisa lebih adil berdasarkan performa dan kerja sama tim, bukan sekadar win rate. Terima kasih. 🙏🔥', 'rating': 3, 'tanggal': '2026-05-19 09:00:00'},
        {'review': 'aku capek montoon aku kalah Mulu terus dapat tim yang gk ngerti mekanik dasar montoon kasih lah gw tim yang betul jangan kasih gw kalah terus,why montoon why?', 'rating': 1, 'tanggal': '2026-05-18 20:15:33'},
        {'review': 'bagus', 'rating': 5, 'tanggal': '2026-05-17 14:00:00'},
        {'review': 'geme bocil', 'rating': 1, 'tanggal': '2026-05-17 13:45:00'},
        {'review': 'game rusak, padahal jaringan nya pas main hijau tapi ngelek" game rusak', 'rating': 1, 'tanggal': '2026-05-16 22:30:00'},
        {'review': 'min kurangi deraksistem nya bikin setres masa miya exp yang bener lah min masa gua yang baru main satu jam udah kalah tiga kali', 'rating': 1, 'tanggal': '2026-05-15 18:55:00'},
        {'review': 'bagusssss banget gamenya suka bangettt sama hero barunya kereeeen!!!', 'rating': 5, 'tanggal': '2026-05-14 10:20:00'},
        {'review': 'matchmaking rusak bgt, rank gw turun terus gara2 dapet tim feeder mulu. fix dong moonton!!', 'rating': 1, 'tanggal': '2026-05-13 21:00:00'},
        {'review': 'grafik nya bagus, gameplay seru, tapi ya itu lag nya ganggu banget kalo sinyal lemah', 'rating': 4, 'tanggal': '2026-05-12 16:45:00'},
        {'review': 'update terbaru malah bikin game makin berat, hp gw jadi panas terus dan baterai boros', 'rating': 2, 'tanggal': '2026-05-11 12:30:00'},
        {'review': 'hero nya banyak pilihan, tapi butuh waktu lama buat unlock semua. harus beli diamond dulu pasti', 'rating': 3, 'tanggal': '2026-05-10 09:15:00'},
        {'review': 'GK ADA YANG LEBIH BURUK DARI SISTEM INI!!! FEEDER DIMANA2 NOOB SEMUA TIM GW!!', 'rating': 1, 'tanggal': '2026-05-09 23:45:00'},
        {'review': 'overall oke lah, masih bisa dimainkan. cuma skin nya mahal2 dikit, p2w banget', 'rating': 3, 'tanggal': '2026-05-08 14:00:00'},
        {'review': 'love this game! sudah main dari season 1 sampe sekarang, masih seru terus 👍😊', 'rating': 5, 'tanggal': '2026-05-07 11:30:00'},
    ]
    df_sample = pd.DataFrame(sample_reviews)
    df_sample.to_csv(CSV_PATH, index=False, encoding='utf-8')
    print(f'   ✅ Sample data ({len(df_sample)} baris) tersimpan di: {CSV_PATH}')

# --------------------
# Load CSV
# --------------------
df = pd.read_csv(CSV_PATH, encoding='utf-8')

# Validasi kolom
required_cols = {'review', 'rating', 'tanggal'}
assert required_cols.issubset(df.columns), f'Kolom wajib tidak ditemukan: {required_cols - set(df.columns)}'

# Parse tanggal
df['tanggal'] = pd.to_datetime(df['tanggal'], errors='coerce')

# Simpan raw copy
df_raw = df.copy()

print(f'\n✅ Dataset berhasil dimuat.')
print(f'   Jumlah baris   : {len(df):,}')
print(f'   Jumlah kolom   : {len(df.columns)}')
print(f'   Kolom          : {list(df.columns)}')
print(f'   Range tanggal  : {df["tanggal"].min()} → {df["tanggal"].max()}')
print(f'\n--- Preview 5 baris pertama ---')
df.head()

---
## 🔍 CELL 4 — Eksplorasi Data Awal (EDA)

In [ ]:
# ============================================================
# CELL 4 — Exploratory Data Analysis (EDA)
# ============================================================

print('='*60)
print('📊 EKSPLORASI DATA AWAL')
print('='*60)

# --- Info Umum ---
print('\n[1] Info Dataset:')
df.info()

print('\n[2] Statistik Deskriptif:')
print(df.describe(include='all'))

# --- Missing Values ---
print('\n[3] Missing Values:')
mv = df.isnull().sum()
print(mv[mv > 0] if mv.sum() > 0 else '   ✅ Tidak ada missing values')

# --- Distribusi Rating ---
print('\n[4] Distribusi Rating:')
print(df['rating'].value_counts().sort_index())

# --- Panjang Review ---
df['review_length'] = df['review'].astype(str).apply(len)
df['word_count']    = df['review'].astype(str).apply(lambda x: len(x.split()))
print('\n[5] Statistik Panjang Review:')
print(df[['review_length', 'word_count']].describe().round(2))

# --- Visualisasi ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('MLBB Review — EDA Overview', fontsize=14, fontweight='bold')

# Rating distribution
rating_counts = df['rating'].value_counts().sort_index()
colors = ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60']
axes[0,0].bar(rating_counts.index, rating_counts.values, color=colors)
axes[0,0].set_title('Distribusi Rating')
axes[0,0].set_xlabel('Rating')
axes[0,0].set_ylabel('Jumlah Review')
for i, (x, y) in enumerate(zip(rating_counts.index, rating_counts.values)):
    axes[0,0].text(x, y + 0.1, str(y), ha='center', fontweight='bold')

# Review length distribution
axes[0,1].hist(df['review_length'], bins=30, color='steelblue', edgecolor='white')
axes[0,1].set_title('Distribusi Panjang Review (karakter)')
axes[0,1].set_xlabel('Panjang (karakter)')
axes[0,1].set_ylabel('Frekuensi')

# Word count distribution
axes[1,0].hist(df['word_count'], bins=30, color='coral', edgecolor='white')
axes[1,0].set_title('Distribusi Jumlah Kata')
axes[1,0].set_xlabel('Jumlah Kata')
axes[1,0].set_ylabel('Frekuensi')

# Review per bulan (jika data cukup)
if df['tanggal'].notna().sum() > 0:
    df['bulan'] = df['tanggal'].dt.to_period('M')
    monthly = df.groupby('bulan').size()
    if len(monthly) > 1:
        monthly.plot(ax=axes[1,1], kind='bar', color='mediumseagreen')
        axes[1,1].set_title('Jumlah Review per Bulan')
        axes[1,1].set_xlabel('Bulan')
        axes[1,1].set_ylabel('Jumlah Review')
        axes[1,1].tick_params(axis='x', rotation=45)
    else:
        axes[1,1].text(0.5, 0.5, 'Data 1 bulan\n(perlu lebih banyak data)', ha='center', va='center', transform=axes[1,1].transAxes)
        axes[1,1].set_title('Review per Bulan')

plt.tight_layout()
plt.savefig('eda_overview.png', bbox_inches='tight')
plt.show()
print('\n✅ Plot EDA tersimpan: eda_overview.png')

---
## 📖 CELL 5 — Kamus Slang & Konfigurasi

In [ ]:
# ============================================================
# CELL 5 — Kamus Slang, Typo, Kata Kasar & Konfigurasi
# ============================================================

# --------------------
# A. Kamus Normalisasi Slang & Typo MLBB
# --------------------
SLANG_DICT = {
    # === Typo nama game/perusahaan ===
    'monton'   : 'moonton', 'muntun': 'moonton', 'munton': 'moonton',
    'montoon'  : 'moonton', 'monton': 'moonton',
    'ml'       : 'mobile legends', 'mlbb': 'mobile legends',
    'geme'     : 'game',

    # === Singkatan umum ===
    'gw'  : 'saya', 'gua': 'saya', 'w': 'saya', 'aku': 'saya',
    'lu'  : 'kamu', 'lo': 'kamu', 'u': 'kamu',
    'yg'  : 'yang', 'yng': 'yang',
    'gak' : 'tidak', 'ngak': 'tidak', 'nggak': 'tidak', 'gk': 'tidak',
    'ga'  : 'tidak', 'enggak': 'tidak', 'nope': 'tidak',
    'bgt' : 'banget', 'bngt': 'banget',
    'dgn' : 'dengan', 'dg': 'dengan',
    'aja' : 'saja',  'aj': 'saja',
    'jg'  : 'juga',  'juga': 'juga',
    'klo' : 'kalau', 'kalo': 'kalau', 'klo': 'kalau',
    'tp'  : 'tapi',  'tpi': 'tapi',
    'krn' : 'karena', 'karna': 'karena',
    'udh' : 'sudah', 'udah': 'sudah', 'dah': 'sudah', 'sdh': 'sudah',
    'blm' : 'belum', 'blum': 'belum',
    'msh' : 'masih', 'msih': 'masih',
    'sm'  : 'sama',  'ama': 'sama',
    'bs'  : 'bisa',  'bsa': 'bisa',
    'skrg': 'sekarang', 'skrang': 'sekarang',
    'utk' : 'untuk', 'tuk': 'untuk', 'buat': 'untuk',
    'dr'  : 'dari',
    'ke'  : 'ke',
    'di'  : 'di',
    'pd'  : 'pada',
    'trs' : 'terus', 'trus': 'terus',
    'mulu': 'melulu', 'mlulu': 'melulu',
    'nih' : 'ini',   'neh': 'ini',
    'sih' : 'sih',
    'lah' : 'lah',
    'dong': 'dong',
    'deh' : 'deh',
    'kek' : 'seperti', 'kyk': 'seperti', 'kyak': 'seperti',
    'emg' : 'memang', 'emang': 'memang', 'mmg': 'memang',
    'mau' : 'mau',
    'biar': 'supaya',
    'dtg' : 'datang',
    'sdg' : 'sedang', 'lg': 'sedang', 'lgi': 'sedang',
    'jd'  : 'jadi',  'jadi': 'jadi',
    'ttg' : 'tentang',
    'hrs' : 'harus',
    'lbh' : 'lebih', 'lebh': 'lebih',
    'dpt' : 'dapat', 'dapet': 'dapat',
    'ntar': 'nanti', 'tar': 'nanti',
    'maen': 'main',  'maein': 'main',
    'hampir': 'hampir',
    'lagi': 'lagi',

    # === MLBB spesifik ===
    'feeder'      : 'pemain buruk', 'feeders': 'pemain buruk',
    'noob'        : 'pemain buruk', 'n00b': 'pemain buruk', 'nub': 'pemain buruk',
    'afk'         : 'tidak bermain',
    'bot'         : 'bot',
    'matchmaking' : 'matchmaking', 'mm': 'matchmaking',
    'rank'        : 'peringkat', 'ranked': 'peringkat',
    'nerf'        : 'diperlemah', 'buff': 'diperkuat',
    'op'          : 'terlalu kuat', 'overpowered': 'terlalu kuat',
    'p2w'         : 'bayar untuk menang',
    'skin'        : 'kostum',
    'diamond'     : 'berlian', 'dia': 'berlian',
    'hero'        : 'karakter',
    'push'        : 'naik rank',
    'lose streak' : 'kalah beruntun',
    'win rate'    : 'tingkat kemenangan',
    'lag'         : 'lag', 'lagi': 'lagi',
    'bug'         : 'bug', 'buggy': 'bermasalah',
    'update'      : 'pembaruan',
    'hack'        : 'curang', 'hacker': 'penipu',
    'carry'       : 'memimpin tim',
    'exp'         : 'pengalaman',
    'offlaner'    : 'penyerang tepi',
    'jungler'     : 'pemburu hutan',

    # === Code-mixing (Inggris → Indonesia) ===
    'good'    : 'bagus', 'great': 'bagus', 'nice': 'bagus', 'best': 'terbaik',
    'bad'     : 'buruk', 'worst': 'terburuk', 'terrible': 'sangat buruk',
    'fix'     : 'perbaiki', 'fixing': 'memperbaiki', 'fixed': 'diperbaiki',
    'please'  : 'tolong', 'pls': 'tolong', 'plz': 'tolong',
    'love'    : 'suka',
    'hate'    : 'benci',
    'fun'     : 'menyenangkan',
    'boring'  : 'membosankan',
    'waste'   : 'membuang',
    'server'  : 'server', 'servers': 'server',
    'support' : 'dukungan',
    'team'    : 'tim',
    'random'  : 'acak',
    'system'  : 'sistem',
    'player'  : 'pemain', 'players': 'pemain',

    # === Kata lebay / ekspresi ===
    'wkwk'   : '', 'wkwkwk': '', 'wkwkwkwk': '',
    'hahaha' : '', 'hahah': '', 'haha': '',
    'hehe'   : '', 'hihi': '',
    'lol'    : '', 'lmao': '', 'omg': '',
    'fomo'   : 'takut ketinggalan',
    'ygy'    : 'ya betul', 'ygw': 'ya sudah',
    'cmn'    : 'cuman', 'cuma': 'hanya', 'cuman': 'hanya',
    'btw'    : 'ngomong-ngomong',
    'imo'    : 'menurut saya',

    # === Typo umum ===
    'bintag'  : 'bintang', 'bintg': 'bintang',
    'seharusny': 'seharusnya',
    'ngelek'  : 'lag', 'ngelekin': 'lag',
    'setres'  : 'stres', 'stress': 'stres',
    'jelas'   : 'jelas',
    'jelass'  : 'jelas',
    'maer'    : 'mahal',
    'kreeeen' : 'keren', 'kereeeen': 'keren',
    'gara2'   : 'gara-gara',
    'min'     : 'developer',
    'pihak'   : 'pihak',
    'kang'    : 'kakak',
    'bosq'    : 'bos',
}

# --------------------
# B. Emoji → Teks (emosional MLBB konteks)
# --------------------
EMOJI_TO_TEXT = {
    '😭': ' sangat sedih ',
    '😢': ' sedih ',
    '😡': ' marah ',
    '🤬': ' sangat marah ',
    '😤': ' kesal ',
    '😠': ' marah ',
    '😒': ' kecewa ',
    '😞': ' kecewa ',
    '😔': ' kecewa ',
    '😩': ' frustasi ',
    '🙁': ' tidak senang ',
    '😐': ' biasa ',
    '😑': ' ekspresi datar ',
    '😊': ' senang ',
    '😄': ' gembira ',
    '😍': ' sangat suka ',
    '🥰': ' suka ',
    '😎': ' keren ',
    '🤩': ' kagum ',
    '😂': ' lucu ',
    '🤣': ' lucu ',
    '👍': ' bagus ',
    '👎': ' buruk ',
    '✅': ' setuju ',
    '❌': ' tidak setuju ',
    '🔥': ' keren ',
    '💔': ' kecewa ',
    '❤️': ' suka ',
    '💪': ' kuat ',
    '🙏': ' mohon ',
    '⭐': ' bintang ',
    '🌟': ' bintang ',
    '💎': ' berlian ',
    '🏆': ' juara ',
    '🚀': ' cepat ',
    '💸': ' mahal ',
    '🎮': ' game ',
}

# --------------------
# C. Stopwords MLBB khusus (jangan dihapus untuk sentimen)
# --------------------
KEEP_WORDS = {
    'bagus', 'baik', 'buruk', 'jelek', 'seru', 'keren', 'bosan',
    'marah', 'kesal', 'senang', 'sedih', 'kecewa', 'stres',
    'frustasi', 'capek', 'lelah', 'puas', 'bangga',
    'lag', 'lambat', 'cepat', 'crash', 'error', 'bug', 'rusak',
    'tidak', 'bukan', 'jangan', 'kurang', 'sangat', 'terlalu',
    'banget', 'sekali', 'lebih', 'paling',
    'menang', 'kalah', 'kuat', 'lemah',
}

# --------------------
# D. Stopwords tambahan MLBB (aman dihapus)
# --------------------
MLBB_STOPWORDS = {
    'mobile', 'legends', 'game', 'aplikasi', 'play', 'store',
    'developer', 'moonton', 'mlbb', 'ml',
}

print('✅ Kamus konfigurasi berhasil dimuat.')
print(f'   Kamus Slang   : {len(SLANG_DICT)} entri')
print(f'   Emoji to Text : {len(EMOJI_TO_TEXT)} entri')
print(f'   Keep Words    : {len(KEEP_WORDS)} kata')

---
## 🔧 CELL 6 — Fungsi Preprocessing

In [ ]:
# ============================================================
# CELL 6 — Semua Fungsi Preprocessing
# ============================================================

# ----------
# Init Sastrawi
# ----------
stemmer_factory  = StemmerFactory()
stemmer          = stemmer_factory.create_stemmer()

sw_factory       = StopWordRemoverFactory()
sastrawi_sw      = set(sw_factory.get_stop_words())
nltk_sw          = set(stopwords.words('indonesian'))

# Gabungkan stopwords & hilangkan kata penting sentimen
ALL_STOPWORDS = (sastrawi_sw | nltk_sw | MLBB_STOPWORDS) - KEEP_WORDS

print(f'Total stopwords gabungan : {len(ALL_STOPWORDS)} kata')


# ─────────────────────────────────────────
# STEP 1 — Emoji Conversion
# ─────────────────────────────────────────
def convert_emoji(text: str) -> str:
    """Konversi emoji ke teks deskriptif (MLBB konteks)."""
    for em, label in EMOJI_TO_TEXT.items():
        text = text.replace(em, label)
    # Hapus emoji yang tidak ada di kamus
    text = emoji.replace_emoji(text, replace=' ')
    return text


# ─────────────────────────────────────────
# STEP 2 — Case Folding
# ─────────────────────────────────────────
def case_fold(text: str) -> str:
    """Ubah semua teks ke lowercase."""
    return text.lower()


# ─────────────────────────────────────────
# STEP 3 — Pembersihan Karakter Noise
# ─────────────────────────────────────────
def clean_chars(text: str) -> str:
    """Bersihkan karakter noise: URL, HTML, angka, tanda baca, whitespace."""
    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # Hapus tag HTML
    text = re.sub(r'<[^>]+>', ' ', text)
    # Hapus mention (@user)
    text = re.sub(r'@\w+', ' ', text)
    # Hapus hashtag
    text = re.sub(r'#\w+', ' ', text)
    # Hapus angka yang berdiri sendiri (bukan '5 bintang', 'season 5', dll)
    # Simpan angka yang menyatu dengan kata konteks
    text = re.sub(r'\b\d+\b', ' ', text)
    # Hapus tanda baca (kecuali strip - pada kata gabung)
    text = re.sub(r'[!"#$%&\'()*+,\./:;<=>?@\[\\\]^_`{|}~]', ' ', text)
    # Hapus karakter non-ASCII yang tersisa
    text = re.sub(r'[^\x00-\x7Fa-zA-Z\s\-]', ' ', text)
    # Normalisasi whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# ─────────────────────────────────────────
# STEP 4 — Normalisasi Kata Berulang
# ─────────────────────────────────────────
def normalize_repeating(text: str) -> str:
    """
    Normalisasi huruf berulang:
    - bagusssss    → bagus
    - jelekkkk     → jelek
    - kereeeen     → keren
    Minimal 3 karakter berulang → 2 (lalu biarkan stemmer bekerja)
    """
    # Normalisasi huruf berulang (3+ → 2)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # Normalisasi kata berulang ditulis dengan tanda (game-game, game"game)
    text = re.sub(r'(\b\w+\b)[\s"\-]+\1', r'\1', text)
    return text


# ─────────────────────────────────────────
# STEP 5 — Normalisasi Slang & Typo
# ─────────────────────────────────────────
def normalize_slang(text: str) -> str:
    """Ganti slang/typo menggunakan SLANG_DICT."""
    words = text.split()
    result = []
    i = 0
    while i < len(words):
        # Cek 2-gram dulu (misal: 'lose streak', 'win rate')
        if i + 1 < len(words):
            bigram = words[i] + ' ' + words[i+1]
            if bigram in SLANG_DICT:
                replacement = SLANG_DICT[bigram]
                if replacement:
                    result.append(replacement)
                i += 2
                continue
        # Cek 1-gram
        w = words[i]
        if w in SLANG_DICT:
            replacement = SLANG_DICT[w]
            if replacement:  # kosong = hapus kata
                result.append(replacement)
        else:
            result.append(w)
        i += 1
    return ' '.join(result)


# ─────────────────────────────────────────
# STEP 6 — Tokenisasi
# ─────────────────────────────────────────
def tokenize(text: str) -> list:
    """Tokenisasi sederhana berbasis spasi (cocok untuk Bahasa Indonesia)."""
    return [w for w in text.split() if len(w) > 1]


# ─────────────────────────────────────────
# STEP 7A — Remove Stopwords (Versi 1: untuk Topic Modeling)
# ─────────────────────────────────────────
def remove_stopwords(tokens: list) -> list:
    """Hapus stopwords (gunakan untuk topic modeling, bukan sentiment)."""
    return [t for t in tokens if t not in ALL_STOPWORDS]


# ─────────────────────────────────────────
# STEP 7B — Stemming (PySastrawi)
# ─────────────────────────────────────────
def stem_tokens(tokens: list) -> list:
    """Stemming dengan PySastrawi."""
    return [stemmer.stem(t) for t in tokens]


# ─────────────────────────────────────────
# PIPELINE UTAMA — Full Preprocessing
# ─────────────────────────────────────────
def preprocess_full(text: str, apply_stopword=False, apply_stem=False) -> str:
    """
    Pipeline preprocessing lengkap.

    Args:
        text         : teks mentah
        apply_stopword: True → hapus stopwords (untuk topic modeling)
        apply_stem   : True → lakukan stemming

    Returns:
        Teks yang sudah dibersihkan (string).
    """
    if pd.isna(text) or str(text).strip() == '':
        return ''
    text = str(text)

    # Step 1: Emoji → Teks
    text = convert_emoji(text)
    # Step 2: Lowercase
    text = case_fold(text)
    # Step 3: Bersihkan karakter noise
    text = clean_chars(text)
    # Step 4: Normalisasi kata berulang
    text = normalize_repeating(text)
    # Step 5: Normalisasi slang & typo
    text = normalize_slang(text)
    # Step 6: Tokenisasi
    tokens = tokenize(text)
    # Step 7A (opsional): Stopword removal
    if apply_stopword:
        tokens = remove_stopwords(tokens)
    # Step 7B (opsional): Stemming
    if apply_stem:
        tokens = stem_tokens(tokens)

    return ' '.join(tokens)


# ----------
# Quick test
# ----------
test_cases = [
    'milih tim nya aneh ngak sesuai sama lawan malah tambah ngak jelass!!',
    'bagusssss banget gamenya suka bangettt sama hero barunya kereeeen!!!',
    'matchmaking rusak bgt, rank gw turun terus gara2 dapet tim feeder mulu 😡😡',
    'game rusak, padahal jaringan nya pas main hijau tapi ngelek" game rusak',
    'Dear Moonton, tolong perbaiki sistem matchmaking yang lebih seimbang 🙏🔥',
]

print('='*65)
print('✅ FUNGSI PREPROCESSING BERHASIL DIMUAT')
print('='*65)
print('\n📝 Quick Test — Sentiment Version (tanpa stopword removal):')
for i, tc in enumerate(test_cases, 1):
    result = preprocess_full(tc, apply_stopword=False, apply_stem=False)
    print(f'\n[{i}] SEBELUM: {tc[:80]}...' if len(tc)>80 else f'\n[{i}] SEBELUM: {tc}')
    print(f'    SESUDAH: {result}')

---
## ⚙️ CELL 7 — Jalankan Preprocessing pada Dataset

In [ ]:
# ============================================================
# CELL 7 — Terapkan Preprocessing ke Seluruh Dataset
# ============================================================

print('⏳ Memproses dataset...')

# Drop baris yang review-nya NaN / kosong
df = df.dropna(subset=['review']).reset_index(drop=True)
df = df[df['review'].astype(str).str.strip() != ''].reset_index(drop=True)

# --- Versi 1: Untuk Sentiment Analysis (tanpa stopword & stem) ---
print('\n[1/2] Membuat versi untuk Sentiment Analysis...')
df['review_clean'] = df['review'].progress_apply(
    lambda x: preprocess_full(x, apply_stopword=False, apply_stem=False)
)

# --- Versi 2: Untuk Topic Modeling (dengan stopword removal + stemming) ---
print('\n[2/2] Membuat versi untuk Topic Modeling...')
df['review_topic'] = df['review'].progress_apply(
    lambda x: preprocess_full(x, apply_stopword=True, apply_stem=True)
)

# Tambah kolom statistik
df['review_clean_length'] = df['review_clean'].apply(len)
df['review_clean_words']  = df['review_clean'].apply(lambda x: len(x.split()))

# Drop review yang jadi kosong setelah preprocessing
df_empty_after = df[df['review_clean'].str.strip() == '']
if len(df_empty_after) > 0:
    print(f'\n⚠️  {len(df_empty_after)} review menjadi kosong setelah preprocessing → dihapus')
    df = df[df['review_clean'].str.strip() != ''].reset_index(drop=True)

print(f'\n✅ Preprocessing selesai!')
print(f'   Total baris valid : {len(df):,}')
print(f'\n--- Perbandingan sebelum vs sesudah ---')
df[['review', 'review_clean', 'review_topic']].head(8)

---
## 🏷️ CELL 8 — Labeling Sentimen dari Rating

In [ ]:
# ============================================================
# CELL 8 — Labeling Sentimen Berdasarkan Rating (Ground Truth)
# ============================================================

def rating_to_sentiment(rating):
    """
    Konversi rating 1–5 ke label sentimen:
    - 1–2  → Negatif
    - 3    → Netral
    - 4–5  → Positif
    """
    if pd.isna(rating):
        return 'Unknown'
    r = int(rating)
    if r <= 2:
        return 'Negatif'
    elif r == 3:
        return 'Netral'
    else:
        return 'Positif'

df['sentiment_label'] = df['rating'].apply(rating_to_sentiment)
df['sentiment_id']    = df['sentiment_label'].map({'Negatif': 0, 'Netral': 1, 'Positif': 2, 'Unknown': -1})

# Distribusi
sentiment_dist = df['sentiment_label'].value_counts()
print('\n📊 Distribusi Sentimen (dari Rating):')
print(sentiment_dist.to_string())
print(f'\n   Proporsi:')
print((sentiment_dist / len(df) * 100).round(2).to_string())

# Pie chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors_map = {'Positif': '#2ecc71', 'Netral': '#f39c12', 'Negatif': '#e74c3c', 'Unknown': '#95a5a6'}

colors_pie = [colors_map.get(c, 'grey') for c in sentiment_dist.index]
axes[0].pie(sentiment_dist.values, labels=sentiment_dist.index, autopct='%1.1f%%',
            colors=colors_pie, startangle=140)
axes[0].set_title('Distribusi Sentimen')

# Rata-rata panjang review per sentimen
avg_len = df.groupby('sentiment_label')['review_clean_words'].mean().sort_values()
bar_colors = [colors_map.get(c, 'grey') for c in avg_len.index]
axes[1].barh(avg_len.index, avg_len.values, color=bar_colors)
axes[1].set_title('Rata-rata Jumlah Kata per Sentimen')
axes[1].set_xlabel('Rata-rata Kata')

plt.tight_layout()
plt.savefig('sentiment_distribution.png', bbox_inches='tight')
plt.show()

print('\n✅ Labeling sentimen selesai. Plot tersimpan: sentiment_distribution.png')

---
## 💬 CELL 9 — VADER Baseline Sentiment Scoring

In [ ]:
# ============================================================
# CELL 9 — VADER Baseline Sentiment (Lexicon-Based)
# ============================================================

vader = SentimentIntensityAnalyzer()

def vader_sentiment(text: str) -> dict:
    """
    Hitung skor VADER.
    Catatan: VADER dioptimasi untuk Inggris; digunakan sebagai baseline.
    Lebih akurat pada review code-mixing.
    """
    scores = vader.polarity_scores(str(text))
    compound = scores['compound']
    if compound >= 0.05:
        label = 'Positif'
    elif compound <= -0.05:
        label = 'Negatif'
    else:
        label = 'Netral'
    return {
        'vader_pos'      : round(scores['pos'], 4),
        'vader_neg'      : round(scores['neg'], 4),
        'vader_neu'      : round(scores['neu'], 4),
        'vader_compound' : round(compound, 4),
        'vader_label'    : label,
    }

print('⏳ Menjalankan VADER scoring...')
vader_results = df['review_clean'].progress_apply(vader_sentiment)
vader_df      = pd.DataFrame(vader_results.tolist())
df            = pd.concat([df, vader_df], axis=1)

# Akurasi VADER vs Ground Truth
match = (df['vader_label'] == df['sentiment_label']).sum()
total = len(df)
print(f'\n✅ VADER selesai.')
print(f'   Akurasi VADER vs Ground Truth (Rating): {match}/{total} = {match/total*100:.1f}%')
print('   (VADER adalah baseline; IndoBERT akan lebih akurat untuk Bahasa Indonesia)')

# Confusion matrix sederhana
from sklearn.metrics import classification_report
# Filter hanya label yang ada di keduanya
valid = df[df['sentiment_label'].isin(['Positif', 'Netral', 'Negatif'])]
print('\n📊 VADER Classification Report:')
print(classification_report(valid['sentiment_label'], valid['vader_label'],
                            labels=['Positif', 'Netral', 'Negatif'], zero_division=0))

df[['review', 'review_clean', 'sentiment_label', 'vader_label', 'vader_compound']].head(10)

---
## 🤖 CELL 10 — IndoBERT Sentiment Analysis

In [ ]:
# ============================================================
# CELL 10 — IndoBERT Sentiment (Deep Learning)
# Menggunakan model fine-tuned untuk sentimen Bahasa Indonesia
# ============================================================

INDOBERT_MODEL = 'mdhugol/indonesia-bert-sentiment-classification'
# Model ini sudah fine-tuned khusus untuk klasifikasi sentimen Bahasa Indonesia
# Label: LABEL_0=Positif, LABEL_1=Netral, LABEL_2=Negatif

INDOBERT_LABEL_MAP = {
    'LABEL_0': 'Positif',
    'LABEL_1': 'Netral',
    'LABEL_2': 'Negatif',
}

print('⏳ Memuat model IndoBERT...')
print(f'   Model: {INDOBERT_MODEL}')
print(f'   Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

try:
    device = 0 if torch.cuda.is_available() else -1
    indobert_pipe = pipeline(
        'text-classification',
        model=INDOBERT_MODEL,
        tokenizer=INDOBERT_MODEL,
        device=device,
        truncation=True,
        max_length=512,
    )

    def indobert_predict(text: str) -> dict:
        """Prediksi sentimen menggunakan IndoBERT."""
        if not text or str(text).strip() == '':
            return {'indobert_label': 'Netral', 'indobert_score': 0.0}
        try:
            result = indobert_pipe(str(text)[:512])[0]
            return {
                'indobert_label': INDOBERT_LABEL_MAP.get(result['label'], result['label']),
                'indobert_score': round(result['score'], 4),
            }
        except Exception as e:
            return {'indobert_label': 'Error', 'indobert_score': 0.0}

    print('\n⏳ Menjalankan IndoBERT inference...')
    print('   (Proses ini butuh beberapa menit tergantung jumlah data)')
    indobert_results = df['review_clean'].progress_apply(indobert_predict)
    indobert_df      = pd.DataFrame(indobert_results.tolist())
    df               = pd.concat([df, indobert_df], axis=1)

    # Akurasi IndoBERT
    valid2 = df[df['sentiment_label'].isin(['Positif', 'Netral', 'Negatif'])]
    match2 = (valid2['indobert_label'] == valid2['sentiment_label']).sum()
    print(f'\n✅ IndoBERT selesai!')
    print(f'   Akurasi IndoBERT vs Ground Truth: {match2}/{len(valid2)} = {match2/len(valid2)*100:.1f}%')
    print('\n📊 IndoBERT Classification Report:')
    print(classification_report(valid2['sentiment_label'], valid2['indobert_label'],
                                labels=['Positif', 'Netral', 'Negatif'], zero_division=0))

except Exception as e:
    print(f'\n⚠️  IndoBERT tidak dapat dimuat: {e}')
    print('   → Pastikan koneksi internet tersedia untuk download model pertama kali.')
    print('   → Melanjutkan tanpa IndoBERT. Hanya VADER yang tersedia.')
    df['indobert_label'] = 'N/A'
    df['indobert_score'] = 0.0

df[['review', 'review_clean', 'sentiment_label', 'vader_label', 'indobert_label', 'indobert_score']].head(10)

---
## 📊 CELL 11 — Analisis & Visualisasi Lanjutan

In [ ]:
# ============================================================
# CELL 11 — Analisis & Visualisasi Lanjutan
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('MLBB Review — Analisis Sentimen Mendalam', fontsize=15, fontweight='bold')
colors_map = {'Positif': '#2ecc71', 'Netral': '#f39c12', 'Negatif': '#e74c3c'}

# --- [0,0] Distribusi Sentimen (Ground Truth) ---
ax = axes[0, 0]
sd = df['sentiment_label'].value_counts()
c  = [colors_map.get(x, 'grey') for x in sd.index]
bars = ax.bar(sd.index, sd.values, color=c, edgecolor='white')
ax.set_title('Distribusi Sentimen (Ground Truth)')
ax.set_ylabel('Jumlah')
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.1,
            str(int(b.get_height())), ha='center', fontweight='bold')

# --- [0,1] VADER vs IndoBERT Comparison ---
ax = axes[0, 1]
methods = ['Ground Truth', 'VADER', 'IndoBERT']
categories = ['Positif', 'Netral', 'Negatif']
x = np.arange(len(categories))
width = 0.25
for i, (method, col) in enumerate([
    ('sentiment_label', 'steelblue'),
    ('vader_label', 'darkorange'),
    ('indobert_label', 'mediumseagreen')
]):
    if col in ['vader_label', 'indobert_label'] and col not in df.columns:
        continue
    counts = [df[df[method]==cat].shape[0] for cat in categories]
    ax.bar(x + i*width, counts, width, label=methods[i], color=col, alpha=0.85)
ax.set_title('Perbandingan Metode Sentimen')
ax.set_xticks(x + width)
ax.set_xticklabels(categories)
ax.legend()
ax.set_ylabel('Jumlah')

# --- [0,2] Distribusi VADER Compound Score ---
ax = axes[0, 2]
if 'vader_compound' in df.columns:
    ax.hist(df['vader_compound'], bins=20, color='steelblue', edgecolor='white')
    ax.axvline(x=0.05, color='green', linestyle='--', label='Positif threshold')
    ax.axvline(x=-0.05, color='red', linestyle='--', label='Negatif threshold')
    ax.set_title('Distribusi VADER Compound Score')
    ax.set_xlabel('Compound Score')
    ax.set_ylabel('Frekuensi')
    ax.legend(fontsize=8)

# --- [1,0] Top Kata Positif ---
ax = axes[1, 0]
pos_text = ' '.join(df[df['sentiment_label']=='Positif']['review_topic'].dropna())
pos_words = Counter(pos_text.split()).most_common(15)
if pos_words:
    w, c = zip(*pos_words)
    ax.barh(list(w), list(c), color='#2ecc71')
    ax.set_title('Top 15 Kata — Review POSITIF')
    ax.set_xlabel('Frekuensi')
    ax.invert_yaxis()

# --- [1,1] Top Kata Negatif ---
ax = axes[1, 1]
neg_text = ' '.join(df[df['sentiment_label']=='Negatif']['review_topic'].dropna())
neg_words = Counter(neg_text.split()).most_common(15)
if neg_words:
    w, c = zip(*neg_words)
    ax.barh(list(w), list(c), color='#e74c3c')
    ax.set_title('Top 15 Kata — Review NEGATIF')
    ax.set_xlabel('Frekuensi')
    ax.invert_yaxis()

# --- [1,2] Tren Rating per Bulan ---
ax = axes[1, 2]
if 'bulan' in df.columns and df['bulan'].notna().sum() > 0:
    monthly_avg = df.groupby('bulan')['rating'].mean()
    if len(monthly_avg) > 1:
        monthly_avg.plot(ax=ax, marker='o', color='royalblue')
        ax.set_title('Tren Rata-rata Rating per Bulan')
        ax.set_xlabel('Bulan')
        ax.set_ylabel('Rata-rata Rating')
        ax.set_ylim(1, 5.5)
        ax.tick_params(axis='x', rotation=45)
    else:
        ax.text(0.5, 0.5, 'Perlu lebih\nbanyak data bulan', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('Tren Rating (butuh lebih banyak data)')
else:
    ax.text(0.5, 0.5, 'Tidak ada data tanggal', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('sentiment_analysis_dashboard.png', bbox_inches='tight')
plt.show()
print('✅ Dashboard tersimpan: sentiment_analysis_dashboard.png')

---
## 💾 CELL 12 — Simpan Dataset Final

In [ ]:
# ============================================================
# CELL 12 — Export Dataset
# ============================================================

# --- Kolom yang disimpan ---
base_cols = ['review', 'rating', 'tanggal', 'review_clean', 'review_topic',
             'review_clean_length', 'review_clean_words', 'sentiment_label', 'sentiment_id']
vader_cols    = [c for c in ['vader_pos','vader_neg','vader_neu','vader_compound','vader_label'] if c in df.columns]
indobert_cols = [c for c in ['indobert_label','indobert_score'] if c in df.columns]

final_cols = base_cols + vader_cols + indobert_cols
df_final   = df[[c for c in final_cols if c in df.columns]]

# --- Simpan CSV lengkap ---
OUT_FULL  = 'mlbb_reviews_processed.csv'
df_final.to_csv(OUT_FULL, index=False, encoding='utf-8-sig')  # utf-8-sig untuk Excel compatibility

# --- Simpan versi minimal untuk modeling ---
OUT_MODEL = 'mlbb_reviews_for_modeling.csv'
modeling_cols = ['review_clean', 'sentiment_label', 'sentiment_id']
if 'indobert_label' in df.columns:
    modeling_cols.append('indobert_label')
df_final[modeling_cols].to_csv(OUT_MODEL, index=False, encoding='utf-8-sig')

# --- Simpan versi topic modeling ---
OUT_TOPIC = 'mlbb_reviews_for_topic.csv'
topic_cols = ['review_topic', 'sentiment_label', 'tanggal']
df_final[[c for c in topic_cols if c in df_final.columns]].to_csv(OUT_TOPIC, index=False, encoding='utf-8-sig')

# --- Ringkasan Statistik ---
print('='*60)
print('📦 OUTPUT FILES')
print('='*60)
print(f'1. {OUT_FULL:<45} → Dataset lengkap')
print(f'2. {OUT_MODEL:<45} → Dataset untuk modeling')
print(f'3. {OUT_TOPIC:<45} → Dataset untuk topic modeling')

print('\n'+'='*60)
print('📊 RINGKASAN PREPROCESSING')
print('='*60)
n_raw  = len(df_raw)
n_final= len(df_final)
print(f'Total review awal          : {n_raw:,}')
print(f'Total review setelah clean : {n_final:,}')
print(f'Review dihapus/kosong      : {n_raw - n_final:,}')
print(f'\nDistribusi Sentimen Final:')
print(df_final['sentiment_label'].value_counts().to_string())

if 'vader_label' in df_final.columns:
    valid = df_final[df_final['sentiment_label'].isin(['Positif', 'Netral', 'Negatif'])]
    vader_acc = (valid['vader_label'] == valid['sentiment_label']).mean()
    print(f'\nAkurasi VADER     : {vader_acc*100:.1f}%')

if 'indobert_label' in df_final.columns and df_final['indobert_label'].ne('N/A').any():
    ib_acc = (valid['indobert_label'] == valid['sentiment_label']).mean()
    print(f'Akurasi IndoBERT  : {ib_acc*100:.1f}%')

print('\n✅ Pipeline selesai! Semua output berhasil disimpan.')

# Preview final
print('\n--- Preview 5 baris dataset final ---')
df_final.head()

---
## 📋 CELL 13 — Ringkasan Noise Detection Report

In [ ]:
# ============================================================
# CELL 13 — Noise Detection & Quality Report
# ============================================================

print('='*65)
print('🔬 LAPORAN DETEKSI NOISE')
print('='*65)

review_raw = df_raw['review'].astype(str)

# 1. Emoji
n_emoji = review_raw.apply(lambda x: bool(emoji.emoji_count(x))).sum()
print(f'\n[1] Review mengandung emoji        : {n_emoji:,} ({n_emoji/len(df_raw)*100:.1f}%)')

# 2. Kata berulang
n_repeat = review_raw.apply(lambda x: bool(re.search(r'(.)\1{2,}', x))).sum()
print(f'[2] Review dengan huruf berulang    : {n_repeat:,} ({n_repeat/len(df_raw)*100:.1f}%)')

# 3. ALL CAPS
n_caps = review_raw.apply(lambda x: x.upper() == x and len(x) > 5).sum()
print(f'[3] Review ALL CAPS                 : {n_caps:,} ({n_caps/len(df_raw)*100:.1f}%)')

# 4. Sangat pendek (< 3 kata)
n_short = review_raw.apply(lambda x: len(x.split()) < 3).sum()
print(f'[4] Review sangat pendek (<3 kata)  : {n_short:,} ({n_short/len(df_raw)*100:.1f}%)')

# 5. URL
n_url = review_raw.apply(lambda x: bool(re.search(r'https?://', x))).sum()
print(f'[5] Review mengandung URL           : {n_url:,} ({n_url/len(df_raw)*100:.1f}%)')

# 6. Duplikasi
n_dup = df_raw.duplicated(subset='review').sum()
print(f'[6] Review duplikat                 : {n_dup:,} ({n_dup/len(df_raw)*100:.1f}%)')

# 7. Slang yang berhasil dinormalisasi
slang_keys = set(SLANG_DICT.keys())
n_slang = review_raw.apply(
    lambda x: any(w in slang_keys for w in x.lower().split())
).sum()
print(f'[7] Review dengan slang/typo        : {n_slang:,} ({n_slang/len(df_raw)*100:.1f}%)')

# Ukuran rata-rata sebelum & sesudah
avg_before = review_raw.apply(len).mean()
avg_after  = df['review_clean'].apply(len).mean()
print(f'\n[8] Rata-rata panjang SEBELUM       : {avg_before:.1f} karakter')
print(f'    Rata-rata panjang SESUDAH       : {avg_after:.1f} karakter')
print(f'    Reduksi                         : {(1 - avg_after/avg_before)*100:.1f}%')

print('\n'+'='*65)
print('✅ Laporan noise selesai.')
print('='*65)